# Spark Connect on EMR-on-EC2 — Local IDE Demo

This notebook connects from your local machine to a Spark Connect session running on an Amazon EMR on EC2 cluster.

**Prereqs on your laptop:**
- Python 3.11 recommended (matches the EMR worker Python if you plan to use UDFs)
- Run Cell 0 below to install/upgrade `boto3` and `pyspark[connect]==4.0.0` in this kernel
- AWS credentials in the default profile (or set `AWS_PROFILE`)


In [5]:
# Cell 0 — Install/upgrade required packages in the current kernel.
# EMR emr.start_session / get_session / get_session_endpoint require botocore>=1.43.24.
# Restart the kernel after this cell finishes (toolbar → Restart).
%pip install --quiet --upgrade "boto3>=1.43.24" "botocore>=1.43.24" "pyspark[connect]==4.0.0"

import boto3, botocore, pyspark
print("boto3   :", boto3.__version__)
print("botocore:", botocore.__version__)
print("pyspark :", pyspark.__version__)
print("\n>>> Now restart the kernel (toolbar → Restart), then run Cell 1.")


[notice] A new release of pip is available: 25.3 -> 26.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
boto3   : 1.43.63
botocore: 1.43.63
pyspark : 4.0.0

>>> Now restart the kernel (toolbar → Restart), then run Cell 1.


In [ ]:
# Cell 1 — Start a Spark Connect session on the cluster (all in one cell)
import boto3, time, urllib.parse

CLUSTER_ID   = "j-0971904CP73A998QOD6"
RUNTIME_ROLE = "arn:aws:iam::111111111111:role/sc-blog-runtime-role"
REGION       = "us-west-2"

emr = boto3.client("emr", region_name=REGION)

# start session (per-user, Runtime-Role isolated)
sess = emr.start_session(
    ClusterId=CLUSTER_ID, Name="local-notebook-demo",
    ExecutionRoleArn=RUNTIME_ROLE,
)
session_id = sess["Id"]
print(f"session: {session_id}")

# poll to IDLE (usually 30–60s)
for i in range(30):
    state = emr.get_session(ClusterId=CLUSTER_ID, SessionId=session_id)["Session"]["State"]
    print(f"  state={state}")
    if state == "IDLE": break
    if state in ("FAILED", "TERMINATED"): raise RuntimeError(f"session {state} before IDLE")
    time.sleep(10)

# get endpoint + auth token
ep = emr.get_session_endpoint(ClusterId=CLUSTER_ID, SessionId=session_id)
host = urllib.parse.urlparse(ep["Endpoint"]).netloc or ep["Endpoint"].replace("https://", "")
token = ep["AuthToken"]
sc_url = (f"sc://{host}:443/"
          f";use_ssl=true"
          f";x-aws-proxy-auth={token}"
          f";authorization={session_id}")
print(f"endpoint: {host}")
print(f"token expires: {ep['AuthTokenExpirationTime']}")

session: is-04833163JV6N0JTAGFGZ
  state=SUBMITTED
  state=STARTING
  state=STARTING
  state=STARTED
  state=IDLE
endpoint: is-04833163JV6N0JTAGFGZ.elasticmapreduce-services.us-west-2.amazonaws.com
token expires: 2026-08-04 00:44:19.032000-07:00


In [2]:
# Cell 2 — Connect via Spark Connect and confirm
from pyspark.sql import SparkSession

spark = SparkSession.builder.remote(sc_url).getOrCreate()
print("Connected. Spark version:", spark.version)
spark.sql("SELECT 'Hello from EMR-on-EC2 via Spark Connect' AS msg, current_timestamp() AS ts").show(truncate=False)

Connected. Spark version: 4.0.2-amzn-0
+---------------------------------------+--------------------------+
|msg                                    |ts                        |
+---------------------------------------+--------------------------+
|Hello from EMR-on-EC2 via Spark Connect|2026-08-04 06:43:22.764915|
+---------------------------------------+--------------------------+



In [ ]:
# Cell 3 — DataFrame + S3 round-trip (cluster does the work, laptop shows the result)
import pyspark.sql.functions as F

path = "s3://sc-blog-demo-us-west-2-111111111111/local-notebook-demo/"

df = (
    spark.range(0, 10_000)
    .withColumn("category", F.when(F.col("id") % 2 == 0, "even").otherwise("odd"))
)

df.groupBy("category").count().show()

df.write.mode("overwrite").parquet(path)
spark.read.parquet(path).filter("id < 20").orderBy("id").show(20)

+--------+-----+
|category|count|
+--------+-----+
|    even| 5000|
|     odd| 5000|
+--------+-----+

+---+--------+
| id|category|
+---+--------+
|  0|    even|
|  1|     odd|
|  2|    even|
|  3|     odd|
|  4|    even|
|  5|     odd|
|  6|    even|
|  7|     odd|
|  8|    even|
|  9|     odd|
| 10|    even|
| 11|     odd|
| 12|    even|
| 13|     odd|
| 14|    even|
| 15|     odd|
| 16|    even|
| 17|     odd|
| 18|    even|
| 19|     odd|
+---+--------+



In [4]:
# Cell 4 — Cleanup: terminate the session on the cluster (releases resources)
emr.terminate_session(ClusterId=CLUSTER_ID, SessionId=session_id)
print(f"terminated session {session_id}")

terminated session is-04833163JV6N0JTAGFGZ
